# SerpentGuard: Milestone 2 dataset pipeline

This notebook rebuilds the SerpentGuard dataset from public sources and produces the processed,
split dataset used from Milestone 3 onward. Run it on Kaggle with **Internet on** (CPU is enough).

| Stage | What it does | Time on Kaggle CPU |
|---|---|---|
| 1. Raw build | Downloads every research-grade iNaturalist photo of every snake species recorded in India, worldwide | about 2.5 h |
| 2. Clean, label, split | Drops tiny files and duplicates, adds taxonomy and WHO labels, assigns grouped splits | about 2 min |
| 3. Process | Verifies each photo, drops cross-split near-duplicates, writes the folder layout and sample grids | about 1 h |
| 4. Upload (optional) | Saves the results as private Kaggle datasets under your own account | about 20 min |

**Fast path:** attach the dataset `puneetbajajiitm/serpentguard-indian-snakes` (raw photos and manifest)
and keep `RUN_STAGE_1 = False`. Stages 2 and 3 then reproduce the published splits exactly, because every
random choice uses a fixed seed.

In [ ]:
import collections, gzip, hashlib, io, json, re, shutil, threading, time, zipfile
import urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from scipy.fft import dctn
from sklearn.model_selection import StratifiedGroupKFold

RUN_STAGE_1 = False      # True: rebuild the raw photos from iNaturalist (about 2.5 h)
SPLIT_SOURCE = "published"  # "published": use the reference split file; "recompute": rebuild it and compare
RAW_HANDLE = "puneetbajajiitm/serpentguard-indian-snakes"
PROCESSED_HANDLE = "puneetbajajiitm/serpentguard-indian-snakes-processed"
SPLITS_HANDLE = "puneetbajajiitm/serpentguard-m2-splits"
UPLOAD = False           # True: upload results as private datasets under your Kaggle account

WORK = Path("/kaggle/tmp/serpentguard")
RAW_ZIP = WORK / "raw" / "images.zip"
RAW_MANIFEST = WORK / "raw" / "manifest.csv"
PREP_DIR = WORK / "prep"
OUT_DIR = WORK / "processed"
for folder in (RAW_ZIP.parent, PREP_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

USER_AGENT = "SerpentGuard-course-project (github.com/Puneet-Bajaj-IITM/Group-5-DS-and-AI-Lab-Project)"
INAT_API = "https://api.inaturalist.org/v1"
INDIA_PLACE_ID = 6681
SERPENTES_TAXON_ID = 85553
SPECIES_PAGE = 500
PER_PAGE = 200
TAXA_BATCH = 30
API_PAUSE_S = 1.1
RETRIES = 5
TIMEOUT_S = 90
PHOTO_SIZE = "medium"
S3_HOST = "inaturalist-open-data.s3.amazonaws.com"
FAST_WORKERS = 16
THROTTLED_WORKERS = 4
THROTTLED_MAX_BYTES_PER_HOUR = 4_500_000_000

MIN_BYTES = 5_000
MIN_SIDE_PX = 100
MIN_PHOTOS_PER_CLASS = 20
TIER1_MIN_PHOTOS = 400
N_FOLDS = 7
TEST_FOLD, VAL_FOLD = 0, 1
SEED = 42
TEST_CREATED_FROM = "2022-01-01"

SPLIT_ORDER = ("train", "val", "test", "test_extra", "unknown_pool")
DEDUP_SPLITS = ("val", "test", "test_extra")
READ_WORKERS = 32
PHASH_SIZE, PHASH_KEEP = 32, 8
GRID_TILE, GRID_COLUMNS, SAMPLES_PER_SPECIES, SAMPLE_SEED = 112, 16, 4, 7

# Species used in the Milestone 1 report (27 species with >= 200 CC-licensed observations in India).
M1_CORE27_TAXON_IDS = {966799, 28868, 53492, 1304708, 29832, 30687, 1304706, 539844, 30264, 29574,
                       32150, 26654, 96224, 31079, 1161474, 1161478, 68530, 64091, 32205, 29550,
                       1161477, 1367260, 1499804, 27650, 29273, 1218631, 32227}

# WHO SEARO 2016 snakebite guidelines, India box (PDF pp. 86-87). Ophiophagus kaalinga was split
# from O. hannah in 2024, so it keeps the category of O. hannah.
WHO_CATEGORY_1 = {"Bungarus caeruleus", "Naja kaouthia", "Naja naja",
                  "Daboia russelii", "Echis carinatus", "Hypnale hypnale"}
WHO_CATEGORY_2 = {"Bungarus fasciatus", "Bungarus niger", "Bungarus sindanus", "Bungarus walli",
                  "Naja oxiana", "Naja sagittifera", "Ophiophagus hannah", "Ophiophagus kaalinga",
                  "Trimeresurus albolabris", "Trimeresurus erythrurus", "Trimeresurus purpureomaculatus",
                  "Craspedocephalus malabaricus", "Craspedocephalus gramineus", "Macrovipera lebetinus"}

## Attach the datasets

`kagglehub.dataset_download` attaches a Kaggle dataset to this notebook by its handle, so nothing has to be
added by hand. The datasets are private: the account running the notebook must be a collaborator
(ask the team for access).

In [ ]:
import kagglehub

import sklearn
print("signed in as:", kagglehub.whoami()["username"], "| scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)
raw_root = None if RUN_STAGE_1 else Path(kagglehub.dataset_download(RAW_HANDLE))
print("raw dataset:", raw_root)

In [ ]:
def log(*parts):
    print(time.strftime("%H:%M:%S"), *parts, flush=True)


def fetch(url: str) -> bytes:
    request = urllib.request.Request(url, headers={"User-Agent": USER_AGENT, "Accept-Encoding": "gzip"})
    for attempt in range(RETRIES):
        try:
            with urllib.request.urlopen(request, timeout=TIMEOUT_S) as response:
                body = response.read()
                return gzip.decompress(body) if response.headers.get("Content-Encoding") == "gzip" else body
        except urllib.error.HTTPError as error:
            if error.code in (403, 404) or attempt == RETRIES - 1:
                raise
        except Exception:
            if attempt == RETRIES - 1:
                raise
        time.sleep(API_PAUSE_S * (attempt + 2))


def fetch_json(url: str) -> dict:
    return json.loads(fetch(url))


def slug(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")

## Stage 1: raw build from iNaturalist

**What:** list every snake species with research-grade observations in India (iNaturalist place 6681,
taxon 85553), then download every research-grade photo of those species from anywhere in the world,
at iNaturalist's `medium` size (at most 500 px on the long side).

**Why:**
- Research grade means at least two identifiers agree and more than two-thirds of them back the species.
  That is our label source.
- Photos from outside India add training data for species that are rare in Indian records (for example
  the king cobra). Test photos stay Indian (Stage 2).
- Taxon ids are used throughout, never names: a name search for "Naja naja" returns a damselfly.

Two sources are left out of this version. The India Biodiversity Portal refuses scripted downloads
(HTTP 403), and attaching the 66 GB SnakeCLEF 2021 mirror stops Kaggle kernels before they start.
Licences are recorded per photo; all-rights-reserved photos are kept for private, non-commercial
course use and flagged in the `license` column.

In [ ]:
class ByteThrottle:
    """Keeps downloads from iNaturalist's own photo server under a polite hourly byte budget."""

    def __init__(self, max_bytes_per_hour):
        self.rate, self.lock, self.started, self.sent = max_bytes_per_hour / 3600, threading.Lock(), time.time(), 0

    def consume(self, n_bytes):
        with self.lock:
            self.sent += n_bytes
            ahead = self.sent / self.rate - (time.time() - self.started)
        if ahead > 0:
            time.sleep(ahead)


class ZipSink:
    """Thread-safe writer: photos stream into one zip so they are stored on disk only once."""

    def __init__(self, path):
        self.zip = zipfile.ZipFile(path, "w", compression=zipfile.ZIP_STORED, allowZip64=True)
        self.lock, self.names = threading.Lock(), set()

    def add(self, name, body):
        with self.lock:
            if name not in self.names:
                self.zip.writestr(name, body)
                self.names.add(name)

    def close(self):
        self.zip.close()


def indian_species():
    species, page = [], 1
    while True:
        data = fetch_json(f"{INAT_API}/observations/species_counts?place_id={INDIA_PLACE_ID}"
                          f"&taxon_id={SERPENTES_TAXON_ID}&quality_grade=research&per_page={SPECIES_PAGE}&page={page}")
        species += [{"name": r["taxon"]["name"], "taxon_id": r["taxon"]["id"]} for r in data["results"]]
        if page * SPECIES_PAGE >= data["total_results"]:
            return species
        page += 1
        time.sleep(API_PAUSE_S)


def inat_records(sp):
    id_above = 0
    while True:
        data = fetch_json(f"{INAT_API}/observations?taxon_id={sp['taxon_id']}&quality_grade=research"
                          f"&per_page={PER_PAGE}&order_by=id&order=asc&id_above={id_above}")
        time.sleep(API_PAUSE_S)
        for obs in data["results"]:
            user, location = obs.get("user") or {}, (obs.get("location") or ",").split(",")
            for position, photo in enumerate(obs.get("photos") or []):
                yield {"source": "inaturalist", "obs_id": obs["id"], "photo_id": photo["id"],
                       "photo_position": position, "species": sp["name"], "taxon_id": sp["taxon_id"],
                       "record_taxon": obs["taxon"]["name"], "observer_id": user.get("id"),
                       "observed_on": obs.get("observed_on"), "created_at": obs.get("created_at"),
                       "in_india": INDIA_PLACE_ID in (obs.get("place_ids") or []),
                       "latitude": location[0], "longitude": location[1], "obscured": obs.get("obscured"),
                       "license": photo.get("license_code") or "all-rights-reserved",
                       "attribution": photo.get("attribution"),
                       "url": photo["url"].replace("/square.", f"/{PHOTO_SIZE}.")}
        if len(data["results"]) < PER_PAGE:
            return
        id_above = data["results"][-1]["id"]


def store(record, sink, throttle):
    entry = f"{slug(record['species'])}/{record['source']}_{record['photo_id']}.jpg"
    try:
        body = fetch(record["url"])
        if throttle:
            throttle.consume(len(body))
        with Image.open(io.BytesIO(body)) as image:
            width, height = image.size
            image.verify()
        sink.add(entry, body)
        return dict(record, file=f"images/{entry}", width=width, height=height, bytes=len(body),
                    sha1=hashlib.sha1(body).hexdigest(), ok=True)
    except Exception as error:
        return dict(record, ok=False, error=repr(error)[:200])


def run_stage_1():
    species = indian_species()
    log(len(species), "Indian species")
    records = [r for sp in species for r in inat_records(sp)]
    log(len(records), "photo records")
    fast = [r for r in records if S3_HOST in r["url"]]
    slow = [r for r in records if S3_HOST not in r["url"]]
    sink, results, lock = ZipSink(RAW_ZIP), [], threading.Lock()

    def pool(batch, workers, throttle):
        with ThreadPoolExecutor(workers) as executor:
            for result in executor.map(lambda r: store(r, sink, throttle), batch):
                with lock:
                    results.append(result)

    jobs = [threading.Thread(target=pool, args=(fast, FAST_WORKERS, None)),
            threading.Thread(target=pool, args=(slow, THROTTLED_WORKERS, ByteThrottle(THROTTLED_MAX_BYTES_PER_HOUR)))]
    for job in jobs:
        job.start()
    for job in jobs:
        job.join()
    sink.close()
    ok = pd.DataFrame([r for r in results if r["ok"]]).drop(columns=["ok"])
    ok.to_csv(RAW_MANIFEST, index=False)
    log(f"{len(ok)} photos saved, {len(results) - len(ok)} failed")


if RUN_STAGE_1:
    run_stage_1()
    raw_manifest_path = RAW_MANIFEST
else:
    raw_manifest_path = raw_root / "manifest.csv"
    log("using attached raw dataset at", raw_root)

## Stage 2: clean, label and split

**Cleaning.** Each step removes photos that would mislead training or evaluation:

| Step | Rule | Why |
|---|---|---|
| Too small | under 5 KB, or shorter side under 100 px | too little detail to show diagnostic scales or markings |
| Conflicting duplicates | the same image bytes (SHA-1) under two different species | the label is unreliable, so every copy is dropped |
| Exact duplicates | the same image bytes more than once | keeps one copy (an Indian one first), so one photo cannot sit in two splits |

**Labels.** Each photo gets its species, genus, family (from iNaturalist's taxonomy), WHO medical-importance
category for India (Category 1, Category 2 or not listed; WHO SEARO 2016), a tier, and a flag for the
27 species of the Milestone 1 report.

- Classes are species with at least 20 clean photos. Tier 1 has at least 400 photos and supports
  full species classification. Tier 2 has 20 to 399 and is for few-shot methods.
- Species under 20 photos form the `unknown_pool`. They are never trained on and test the "unknown" answer.

**Splits.** `StratifiedGroupKFold` with 7 folds, seed 42, grouped by **photographer** (iNaturalist user)
and stratified by species. Fold 0 is the test fold, fold 1 is validation, folds 2 to 6 are training.

- Grouping by photographer keeps the same camera, place and individual snake out of both training and
  evaluation. It also groups every photo of an observation.
- Within the test fold, `test` keeps photos taken in India and uploaded on or after 1 January 2022. The
  2022 cutoff means the published global model (trained on SnakeCLEF 2021 data) has never seen them.
  The rest of the test fold becomes `test_extra`, which is never moved into training, so no photographer
  crosses splits.

**Reference split.** The exact fold each photographer lands in depends on the scikit-learn version
(`StratifiedGroupKFold` shuffling changed between releases). Our reference split was made with scikit-learn
1.9.1 and is published as `splits.csv` in `puneetbajajiitm/serpentguard-m2-splits`. With
`SPLIT_SOURCE = "published"` the notebook uses that file. With `"recompute"` it rebuilds the split with the
code below and reports how many photos land in the same split.

In [ ]:
def fetch_families(taxon_ids):
    families = {}
    for start in range(0, len(taxon_ids), TAXA_BATCH):
        batch = ",".join(str(t) for t in taxon_ids[start:start + TAXA_BATCH])
        for taxon in fetch_json(f"{INAT_API}/taxa/{batch}")["results"]:
            families[taxon["id"]] = next((a["name"] for a in taxon.get("ancestors", []) if a.get("rank") == "family"), None)
        time.sleep(API_PAUSE_S)
    return families


def who_category(species):
    return "1" if species in WHO_CATEGORY_1 else "2" if species in WHO_CATEGORY_2 else "not_listed"


def clean(df):
    df = df.copy()
    df["drop_reason"] = ""
    small = (df["bytes"] < MIN_BYTES) | (df[["width", "height"]].min(axis=1) < MIN_SIDE_PX)
    df.loc[small, "drop_reason"] = "too_small"
    species_per_hash = df.groupby("sha1")["species"].nunique()
    conflicting = set(species_per_hash[species_per_hash > 1].index)
    df.loc[df["sha1"].isin(conflicting) & (df["drop_reason"] == ""), "drop_reason"] = "duplicate_conflicting_species"
    order = df.sort_values(["in_india", "obs_id"], ascending=[False, True])
    repeat = set(order.index[order.duplicated("sha1", keep="first")])
    df.loc[df.index.isin(repeat) & (df["drop_reason"] == ""), "drop_reason"] = "exact_duplicate"
    return df[df["drop_reason"] == ""].drop(columns="drop_reason"), df[df["drop_reason"] != ""]


def assign_splits(df):
    folds = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_of = np.full(len(df), -1)
    for fold, (_, idx) in enumerate(folds.split(df, df["species"], groups=df["observer_id"])):
        fold_of[idx] = fold
    split = np.where(fold_of == TEST_FOLD, "test", np.where(fold_of == VAL_FOLD, "val", "train"))
    eligible = (df["in_india"] & (df["created_at"] >= TEST_CREATED_FROM)).to_numpy()
    return pd.Series(np.where((split == "test") & ~eligible, "test_extra", split), index=df.index)


raw = pd.read_csv(raw_manifest_path, dtype={"obs_id": str, "photo_id": str, "observer_id": str, "sha1": str})
raw["in_india"] = raw["in_india"].astype(str) == "True"
raw["created_at"] = raw["created_at"].fillna("").astype(str)
kept, dropped = clean(raw)
counts = kept["species"].value_counts()
classes = sorted(counts[counts >= MIN_PHOTOS_PER_CLASS].index)
data = kept[kept["species"].isin(classes)].copy()
unknown = kept[~kept["species"].isin(classes)].copy()
data["split"] = assign_splits(data)
unknown["split"] = "unknown_pool"

families = fetch_families(sorted({int(t) for t in raw["taxon_id"].dropna().unique()}))
splits = pd.concat([data, unknown])
splits["genus"] = splits["species"].str.split().str[0]
splits["family"] = splits["taxon_id"].map(lambda t: families.get(int(t)) if pd.notna(t) else None)
splits["who_category"] = splits["species"].map(who_category)
tier_counts = data["species"].value_counts()
splits["tier"] = [("unknown_pool" if s == "unknown_pool" else "tier1" if tier_counts[sp] >= TIER1_MIN_PHOTOS else "tier2")
                  for s, sp in zip(splits["split"], splits["species"])]
splits["m1_core27"] = splits["taxon_id"].map(lambda t: pd.notna(t) and int(t) in M1_CORE27_TAXON_IDS)
splits["cc_licensed"] = splits["license"] != "all-rights-reserved"
splits["path"] = [f"{s}/{slug(sp)}/{Path(f).name}" for s, sp, f in zip(splits["split"], splits["species"], splits["file"])]
published = pd.read_csv(Path(kagglehub.dataset_download(SPLITS_HANDLE)) / "splits.csv",
                        dtype={"obs_id": str, "photo_id": str, "observer_id": str, "sha1": str})
if SPLIT_SOURCE == "published" and not RUN_STAGE_1:
    agreement = splits.set_index("path")["split"].reindex(published["path"]).eq(published["split"].values).mean()
    print(f"recomputed split agrees with the published split for {agreement:.1%} of photos; using the published split")
    splits = published
splits.to_csv(PREP_DIR / "splits.csv", index=False)
dropped.to_csv(PREP_DIR / "excluded.csv", index=False)

block = splits[splits["split"] != "unknown_pool"]["split"].replace({"test_extra": "test"})
labelled = splits[splits["split"] != "unknown_pool"].assign(block=block)
for key in ("observer_id", "obs_id", "sha1"):
    print(f"{key} in more than one of train/val/test fold:", int(labelled.groupby(key)["block"].nunique().gt(1).sum()))
print("dropped:", dropped["drop_reason"].value_counts().to_dict())
print("photos per split:", splits["split"].value_counts().to_dict())
print("classes:", len(classes), "| tier 1:", int((tier_counts >= TIER1_MIN_PHOTOS).sum()))

## Stage 3: process into the final layout

**What:** read every kept photo, check its SHA-1 against the manifest, compute a 64-bit perceptual hash
(pHash: 32 x 32 grayscale, 2-D DCT, top-left 8 x 8 coefficients against their median), and write it to
`<split>/<species>/<file>.jpg`.

**Why:**
- Splits are written in the order train, val, test, test_extra. A val or test photo is dropped if its pHash
  already appeared earlier. That removes resized or recompressed copies that SHA-1 cannot catch, which
  would otherwise inflate scores.
- Photos stay at iNaturalist's `medium` size (at most 500 px). Resizing, cropping and normalisation are
  left to each model's data loader, because the encoders compared in Milestone 3 expect different input
  sizes (224, 256 or 384 px).
- The folder layout works directly with `torchvision.datasets.ImageFolder` or Keras
  `image_dataset_from_directory`, while `metadata.csv` carries genus, family and WHO labels for
  multi-task heads.

In [ ]:
def phash(body):
    with Image.open(io.BytesIO(body)) as image:
        gray = np.asarray(image.convert("L").resize((PHASH_SIZE, PHASH_SIZE), Image.LANCZOS), dtype=np.float32)
    low = dctn(gray, norm="ortho")[:PHASH_KEEP, :PHASH_KEEP].flatten()
    bits = low > np.median(low[1:])
    return f"{int(''.join('1' if b else '0' for b in bits), 2):016x}"


def raw_reader():
    if RUN_STAGE_1:
        archive = zipfile.ZipFile(RAW_ZIP)
        return lambda file: archive.read(file.removeprefix("images/"))
    candidates = lambda file: (raw_root / "images" / file, raw_root / file)
    return lambda file: next(p for p in candidates(file) if p.exists()).read_bytes()


read_raw = raw_reader()
sink = ZipSink(OUT_DIR / "data.zip")
status, seen = {}, set()


def load(row):
    try:
        body = read_raw(row["file"])
        return row["path"], body, hashlib.sha1(body).hexdigest() == row["sha1"], phash(body)
    except Exception as error:
        return row["path"], None, False, repr(error)[:200]


for split in SPLIT_ORDER:
    rows = splits[splits["split"] == split].to_dict("records")
    log(split, len(rows), "photos")
    with ThreadPoolExecutor(READ_WORKERS) as executor:
        for path, body, sha_ok, ph in executor.map(load, rows):
            if body is None:
                status[path] = ("read_error", None, False)
            elif split in DEDUP_SPLITS and ph in seen:
                status[path] = ("near_duplicate", ph, sha_ok)
            else:
                sink.add(path, body)
                seen.add(ph)
                status[path] = ("kept", ph, sha_ok)
sink.close()

splits["status"] = splits["path"].map(lambda p: status[p][0])
splits["phash"] = splits["path"].map(lambda p: status[p][1])
splits["sha1_verified"] = splits["path"].map(lambda p: status[p][2])
final = splits[splits["status"] == "kept"].drop(columns="status")
final.to_csv(OUT_DIR / "metadata.csv", index=False)
print(splits["status"].value_counts().to_dict(), "| SHA-1 mismatches among kept:", int((~final["sha1_verified"]).sum()))

In [ ]:
labelled = final[final["split"] != "unknown_pool"]
table = labelled.pivot_table(index="species", columns="split", values="photo_id", aggfunc="count", fill_value=0)
table = table.reindex(columns=["train", "val", "test", "test_extra"], fill_value=0)
table = table.join(labelled.drop_duplicates("species").set_index("species")[["genus", "family", "who_category", "tier", "m1_core27", "taxon_id"]])
table["total"] = table[["train", "val", "test", "test_extra"]].sum(axis=1)
table = table.sort_values("total", ascending=False)
table.insert(0, "class_id", range(len(table)))
table.to_csv(OUT_DIR / "classes.csv")

block = labelled["split"].replace({"test_extra": "test"})
for key in ("observer_id", "obs_id", "sha1", "phash"):
    print(f"{key} in more than one of train/val/test fold:", int(labelled.assign(b=block).groupby(key)["b"].nunique().gt(1).sum()))
table.head(10)

## Stage 4: sample grids and optional upload

The grids show one training photo per class, and four per species for the Milestone 1 core and the
WHO-listed species. Set `UPLOAD = True` to save the processed folder as a **private** dataset under the
account running the notebook.

In [ ]:
def grid(species, per_species, path):
    rng, tiles = np.random.default_rng(SAMPLE_SEED), []
    with zipfile.ZipFile(OUT_DIR / "data.zip") as archive:
        for name in species:
            pool = final[(final["species"] == name) & (final["split"] == "train")]
            if pool.empty:
                continue
            for p in pool.iloc[rng.choice(len(pool), size=min(per_species, len(pool)), replace=False)]["path"]:
                with Image.open(io.BytesIO(archive.read(p))) as im:
                    tiles.append(im.convert("RGB").resize((GRID_TILE, GRID_TILE)))
    columns = per_species if per_species > 1 else GRID_COLUMNS
    canvas = Image.new("RGB", (columns * GRID_TILE, -(-len(tiles) // columns) * GRID_TILE), "white")
    for i, tile in enumerate(tiles):
        canvas.paste(tile, ((i % columns) * GRID_TILE, (i // columns) * GRID_TILE))
    canvas.save(path, quality=90)
    return canvas


figures = OUT_DIR / "figures"
figures.mkdir(exist_ok=True)
grid(list(table.index), 1, figures / "grid_all_classes.jpg")
grid(list(table[table["m1_core27"]].index), SAMPLES_PER_SPECIES, figures / "grid_core27.jpg")
grid(list(table[table["who_category"] != "not_listed"].index), SAMPLES_PER_SPECIES, figures / "grid_who_listed.jpg")

if UPLOAD:
    import kagglehub
    owner = kagglehub.whoami()["username"]
    kagglehub.dataset_upload(f"{owner}/serpentguard-indian-snakes-processed", str(OUT_DIR), version_notes="rebuilt with m2_dataset_pipeline")

## Use the processed dataset

Some species have no photos in `val/` or `test/` (they are rarely photographed in India), so those splits
have fewer species folders than `train/`. Plain `torchvision.datasets.ImageFolder` numbers classes by the
folders it finds, which would give the same species different numbers in different splits and silently
mislabel every photo after the first missing folder. Always take class ids from `classes.csv`
(`class_id`), which is the same for every split. The dataset class below does that from `metadata.csv`;
genus, family and WHO labels come from the same row.

In [ ]:
processed_root = Path(kagglehub.dataset_download(PROCESSED_HANDLE))
meta = pd.read_csv(processed_root / "metadata.csv")
classes_table = pd.read_csv(processed_root / "classes.csv")
print(meta["split"].value_counts())
print(len(classes_table), "classes;", classes_table["tier"].value_counts().to_dict())

from torch.utils.data import Dataset

CLASS_ID = dict(zip(classes_table["species"], classes_table["class_id"]))
UNKNOWN_CLASS_ID = -1


class SerpentGuardSplit(Dataset):
    """One split of the processed dataset. Class ids come from classes.csv, so they match across splits."""

    def __init__(self, root, split, transform=None):
        rows = meta[meta["split"] == split]
        self.paths = [root / "data" / p for p in rows["path"]]
        self.labels = [CLASS_ID.get(s, UNKNOWN_CLASS_ID) for s in rows["species"]]
        self.extra = rows[["genus", "family", "who_category"]].to_dict("records")
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        image = Image.open(self.paths[i]).convert("RGB")
        return (self.transform(image) if self.transform else image), self.labels[i]


for split in ("train", "val", "test", "test_extra"):
    folders = sum(1 for p in (processed_root / "data" / split).iterdir() if p.is_dir())
    print(f"{split:10s} {len(SerpentGuardSplit(processed_root, split)):7d} photos, {folders:3d} of {len(CLASS_ID)} species folders")
classes_table.head(10)